<a href="https://colab.research.google.com/github/Nurana100/flyrank-ml-starter-nurana/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nurana100/flyrank-ml-starter-nurana/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
One row in the source table (fact_content_daily_performance) is one page-day: one
content_hash_id × client_hash_id × report_date. My lane analyzes at the page level,
so I aggregate this up to one row = one content page per client, summarizing a
90-day window ending at the last day of the mid-panel month 2026-03
(2025-12-31 to 2026-03-31), split into a last-30 / prev-30 comparison the same way
trend_direction is built in the starter CSV.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass, duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM {FACT_MAR}
    GROUP BY 1,2,3
    HAVING c > 1
    LIMIT 5
""").df()



Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,c


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
Feature: gsc_impressions, gsc_clicks, gsc_avg_position aggregated over the prev-30
slice only (days 31-60 back from month end) — never last-30, since last-30 is what
the label is computed from. From fact_content_query_90d: content_visible_query_count,
rare_impressions_share, anonymized_impressions_share (query-mix signals, ANY_VALUE()'d
per content item).

Label / proxy: a decline flag built the same way as starter notebook 03 —
imp_last30 < 0.8 * imp_prev30 — computed from gsc_impressions in the last-30 slice.
This is a proxy on the same signal as the feature window, so it must never leak into
features.

Context: client_hash_id, content_hash_id, report_date — grouping, joining, and
client-holdout splits only, never features.

Excluded: gsc_impressions/gsc_clicks from the last-30 slice (label-derived, would be
leakage if used as a feature); any GA4 columns where ga4_data_available is not TRUE
(zero-filled or NULL — not real zeros, not real measurements).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"DESCRIBE SELECT * FROM {FACT_MAR}").df()


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
Three verification queries below, run against month=2026-03:
1. Grain: one row per client x content x report_date (checked above in section 1 —
   zero rows returned confirms the grain holds).
2. Counts + date span: total rows, distinct clients, distinct content items, and the
   actual min/max report_date for this partition.
3. Availability: filtering ga4_data_available with IS TRUE (not = TRUE, since the
   flag is three-valued: TRUE / FALSE / NULL) and showing how many rows survive.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {FACT_MAR}
""").df()



,n_rows,n_clients,n_content,min_d,max_d
0,9841378,55,331437,2026-03-01,2026-03-31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
Unbalanced panel: history depth varies per client (dim_clients.gsc_data_start) — a
global 90-day window can silently exclude newer clients or waste history for older
ones. Per-client windows would be more honest than one global calendar window.

ga4_data_available is three-valued (TRUE/FALSE/NULL, not just TRUE/FALSE) — IS TRUE
filtering is required; plain "= TRUE" or a negation silently mishandles the NULLs.

fact_content_query_90d's window overlaps the last ~3 months of the panel, so its
signals can't safely be used as features without checking alignment against this
month's last-30 label window first.

Named limitation of my slice: [run section 3's availability query and fill this in
with your real number, e.g. "X% of month=2026-03 rows have ga4_data_available NOT
TRUE, so decline detection for those pages relies on GSC signals alone — no
engagement data to corroborate."]


In [ ]:
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_start,
           MAX(gsc_data_start) AS latest_start,
           COUNT(*) AS n_clients
    FROM {DIM_CLIENTS}
""").df()

,earliest_start,latest_start,n_clients
0,2025-01-27,2026-06-02,104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.